In [1]:
import httpx
import requests
import brotli
import json
import time
import pandas as pd  # Ensure Pandas is installed
import sqlite3
import os
from datetime import datetime, date
from requests.adapters import HTTPAdapter
from requests.packages.urllib3.util.retry import Retry
BASE_DIR = "/home/shail/stockshortlisting"
BASE_DIR_db = "/home/shail/db"


## Growth DB

In [22]:
growdb = os.path.join(BASE_DIR_db, "growth_nse_db.db")
gr_conn = sqlite3.connect(growdb)
query_gr = f"SELECT * FROM cm_growth "
grow_df = pd.read_sql_query(query_gr, gr_conn)
grow_df.head()

,Date,Trdstocks,NoTrades,Tval,Tvol,Avg
0,2026-02-27,3243,34269344,144962.74,54541.88,265.78
1,2026-02-26,3265,30748283,108665.76,45182.34,240.50
2,2026-02-25,3291,31214021,108503.78,41055.84,264.28
3,2026-02-24,3279,32472168,113004.60,49224.90,229.57
4,2026-02-23,3288,31051512,101052.48,49124.95,205.71


In [23]:
grow_df.shape

(560, 6)

In [24]:
growdb = os.path.join(BASE_DIR_db, "growth_nse_db.db")
gr_conn = sqlite3.connect(growdb)
query_gr = f"SELECT * FROM fno_growth ORDER BY date DESC "
fno_df = pd.read_sql_query(query_gr, gr_conn)
fno_df.head()

,Date,idxFuvol,IdxFuVal,eqFuvol,eqFuval,idxOpvol,idxOpval,idxOppremval,idxOpPCR,eqOpvol,eqOpval,eqOppremval,eqOpPCR,fnoTvol,fnoTval,TpremVal,fnoPCR
0,2026-02-26,87935,14974.94,1187313,82255.43,71539015,11929791.91,40495.09,1.03,4429355,315208.30,6831.99,0.53,77243618,12342230.58,144557.45,0.99
1,2026-02-26,87935,14974.94,1187313,82255.43,71539015,11929791.91,40495.09,1.03,4429355,315208.30,6831.99,0.53,77243618,12342230.58,144557.45,0.99
2,2026-02-25,130585,22287.35,1288596,86906.43,70108735,11723257.95,46575.46,1.00,5337350,375205.39,8430.60,0.54,76865266,12207657.12,164199.84,0.96
3,2026-02-25,130585,22287.35,1288596,86906.43,70108735,11723257.95,46575.46,1.00,5337350,375205.39,8430.60,0.54,76865266,12207657.12,164199.84,0.96
4,2026-02-24,290782,49399.34,3818932,249370.10,462926472,78118228.69,94539.55,1.11,7613954,515292.98,7481.46,0.72,474650140,78932291.11,400790.45,1.10


In [18]:
spurdb = os.path.join(BASE_DIR_db, "fuopspur.db")
conn = sqlite3.connect(spurdb)
query_spur = f"SELECT * FROM spur WHERE symbol = '{stock}'ORDER BY Date DESC , Time DESC"
df = pd.read_sql_query(query_spur, conn)

In [19]:
spurdb = os.path.join(BASE_DIR_db, "fuopspur.db")
conn = sqlite3.connect(spurdb)
query_spur = f"SELECT * FROM spur ORDER BY Date DESC , Time DESC"
spurdf = pd.read_sql_query(query_spur, conn)

In [20]:
print (spurdf.shape)
spurdf.head()

(2283774, 13)


,symbol,latestOI,prevOI,changeInOI,avgInOI,volume,futValue,optValue,total,premValue,underlyingValue,Time,Date
0,SAMMAANCAP,49497,32772,16725,51.03,44677,2.351048e+05,6164993721,2.361959e+05,1.091102e+03,152,11:08,2026-02-24
1,FINNIFTY,101873,68078,33795,49.64,829721,5.081332e+03,1413746538486,2.546712e+04,2.038578e+04,28345,11:08,2026-02-24
2,NIFTY,12939409,9222783,3716626,40.30,98032183,1.302778e+06,162872140249691,3.996709e+06,2.693931e+06,25468,11:08,2026-02-24
3,BANKNIFTY,2174464,1554452,620012,39.89,12564511,3.041855e+05,23055955343383,6.874464e+05,3.832609e+05,61191,11:08,2026-02-24
4,MIDCPNIFTY,323244,243455,79789,32.77,1814717,9.761435e+04,2923874038029,1.607447e+05,6.313030e+04,13425,11:08,2026-02-24


## Getting only Bank Nifty Spur data.

In [21]:

# Assuming your DataFrame is named df
symbols_to_keep = [
    "HDFCBANK", "ICICIBANK", "SBIN", "KOTAKBANK", "AXISBANK",
    "FEDERALBNK", "CANBK", "BANKBARODA", "UNIONBANK", "PNB",
    "IDFCFIRSTB", "AUBANK", "INDUSINDBK", "YESBANK"
]

filtered_df = spurdf[spurdf["symbol"].isin(symbols_to_keep)]


In [22]:
bn = filtered_df.copy()

In [23]:
print(bn.shape)
bn.head()

(145944, 13)


,symbol,latestOI,prevOI,changeInOI,avgInOI,volume,futValue,optValue,total,premValue,underlyingValue,Time,Date
29,KOTAKBANK,141687,135376,6311,4.66,24122,96259.16800,11186046220,97283.08020,1023.91220,430,11:08,2026-02-24
31,IDFCFIRSTB,109282,104473,4809,4.60,64635,79696.47444,35556852026,86815.30221,7118.82776,70,11:08,2026-02-24
41,YESBANK,59117,57093,2024,3.55,12140,41713.61829,3928116600,42396.26329,682.64500,20,11:08,2026-02-24
80,HDFCBANK,709336,696466,12870,1.85,106372,193006.20625,35473402305,196172.22680,3166.02055,918,11:08,2026-02-24
87,INDUSINDBK,80253,78938,1315,1.67,19169,60061.18706,6486593841,60786.40547,725.21841,924,11:08,2026-02-24


In [25]:
icici = bn[bn['symbol'] == 'ICICIBANK']
idfc = bn[bn['symbol'] == 'IDFCFIRSTB']
idfc.head()

,symbol,latestOI,prevOI,changeInOI,avgInOI,volume,futValue,optValue,total,premValue,underlyingValue,Time,Date
31,IDFCFIRSTB,109282,104473,4809,4.60,64635,79696.47444,35556852026,86815.30221,7118.82776,70,11:08,2026-02-24
241,IDFCFIRSTB,109236,104473,4763,4.56,64363,79494.72000,35394604266,86584.14416,7089.42416,70,11:06,2026-02-24
453,IDFCFIRSTB,109238,104473,4765,4.56,63795,79066.01652,35048689802,86103.28179,7037.26527,70,11:04,2026-02-24
665,IDFCFIRSTB,109158,104473,4685,4.48,63264,78528.61003,34739429674,85506.29178,6977.68174,70,11:02,2026-02-24
877,IDFCFIRSTB,109166,104473,4693,4.49,62447,77368.17676,34306022094,84263.72095,6895.54419,70,11:00,2026-02-24


In [26]:
icici.head(21)

,symbol,latestOI,prevOI,changeInOI,avgInOI,volume,futValue,optValue,total,premValue,underlyingValue,Time,Date
92,ICICIBANK,255769,251961,3808,1.51,42143,121697.44188,29347025659,123631.97847,1934.53659,1389,11:08,2026-02-24
324,ICICIBANK,254475,251961,2514,1.00,41789,120671.07577,29097783246,122565.42823,1894.35246,1389,11:06,2026-02-24
537,ICICIBANK,254445,251961,2484,0.99,40310,112257.32539,28485429868,114080.83407,1823.50868,1389,11:04,2026-02-24
749,ICICIBANK,254257,251961,2296,0.91,39091,105194.12295,27995167564,106974.68859,1780.56564,1389,11:02,2026-02-24
960,ICICIBANK,254272,251961,2311,0.92,38622,103816.64377,27670412154,105563.98531,1747.34154,1389,11:00,2026-02-24
1172,ICICIBANK,254208,251961,2247,0.89,38143,102770.99294,27305860960,104487.41254,1716.41960,1389,10:58,2026-02-24
1387,ICICIBANK,254087,251961,2126,0.84,37713,101286.61452,27031665374,102984.49826,1697.88374,1389,10:56,2026-02-24
1594,ICICIBANK,254225,251961,2264,0.90,36969,99948.22838,26432146741,101594.27579,1646.04741,1389,10:54,2026-02-24
1815,ICICIBANK,253753,251961,1792,0.71,36029,93474.02281,26151851593,95091.91874,1617.89593,1390,10:52,2026-02-24
2027,ICICIBANK,253451,251961,1490,0.59,34782,90104.70105,25253942161,91619.64266,1514.94161,1390,10:50,2026-02-24


In [5]:
idxfudb = os.path.join(BASE_DIR_db, "idxfutures.db")
idx_conn = sqlite3.connect(idxfudb)
query_idx = f"SELECT * FROM futuresIndex WHERE Stock = '{stock}'ORDER BY Date DESC , Time DESC"
idxdf = pd.read_sql_query(query_idx, idx_conn)
idxdf.head()

,Stock,Date,Time,Ltp_fu,PrChg_fu,%Prchg_fu,Tsell_fu,Tbuy_fu,Tvol_fu,Tvalue_fu,fuOI
0,BANKNIFTY,2026-02-23,15:53,61219,39.8,0.07,32250,28170,22715,41718764241,28727
1,BANKNIFTY,2026-02-23,15:32:03,61219,39.8,0.07,32250,28170,22715,41718764241,28727
2,BANKNIFTY,2026-02-23,15:30:03,61208,28.8,0.05,29550,23220,22642,41584725133.8,28727
3,BANKNIFTY,2026-02-23,15:28:03,61225.2,46,0.08,36060,33840,22495,41314728658.5,28727
4,BANKNIFTY,2026-02-23,15:26:03,61237,57.8,0.09,41220,31800,22380,41103510840,28777


## Futures and Options Live Date Processing

In [6]:

starttime = datetime.now()
print("\2\n"+ " ******************  Script run started  at ...   ", starttime)

dbpath = os.path.join(BASE_DIR_db, "fuopspur.db")
conn = sqlite3.connect(dbpath)
current_date = starttime.strftime('%Y-%m-%d')
#n50fu_url = "https://www.nseindia.com/api/liveEquity-derivatives?index=nse50_fut"
#n50op_url = "https://www.nseindia.com/api/liveEquity-derivatives?index=nse50_opt"
#nbfu_url = "https://www.nseindia.com/api/liveEquity-derivatives?index=nifty_bank_fut"
#nbop_url = "https://www.nseindia.com/api/liveEquity-derivatives?index=nifty_bank_opt"

api_url = "https://www.nseindia.com/api/liveEquity-derivatives?index=nse50_fut"

# Headers to mimic a real browser request
headers = {
    "User-Agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36",
    "Accept": "application/json, text/plain, */*",
    "Referer": "https://www.nseindia.com/market-data/oi-spurts",
    "Connection": "keep-alive",
    "Accept-Encoding": "gzip, deflate, br"
	}

# Main NSE site to retrieve cookies
main_url = "https://www.nseindia.com"

# Start session with httpx
with httpx.Client() as client:
    try:
	    response = client.get(main_url, headers=headers)
	    response.raise_for_status()
	    cookies = dict(client.cookies)  # Extract cookies
	    time.sleep(3)  # Wait briefly before the next request	
	    response2 = client.get(api_url, headers=headers, cookies=cookies)
	    response2.raise_for_status()
	    data = json.loads(response2.text)
	    df = pd.json_normalize(data['data'])
	    current_time = starttime.strftime('%H:%M')
	    df['Time'] = current_time
	    df['Date'] = current_date
	    #print(df.shape)
    except httpx.HTTPStatusError as e:
        print(f"Error fetching data: {e}")
#df.to_sql('spur', conn, if_exists='append', index=False)
conn.close()
#df.to_sql('spur', conn, if_exists='append', index=False)
#bank = df[nfcolumns]
endtime = datetime.now()
elapsed = endtime - starttime

# total seconds as float
total_seconds = elapsed.total_seconds()
# minutes and seconds
minutes = int(total_seconds // 60)
seconds = int(total_seconds % 60)
milliseconds = int((total_seconds - int(total_seconds)) * 1000)
print(f"The time taken to complete the task is {minutes}:{seconds:02d}.{milliseconds:03d}")


 ******************  Script run started  at ...    2026-02-23 16:44:10.223918
The time taken to complete the task is 0:03.487


In [7]:
df.head()

,underlying,identifier,instrumentType,instrument,contract,expiryDate,optionType,strikePrice,lastPrice,change,...,closePrice,volume,totalTurnover,value,premiumTurnOver,underlyingValue,openInterest,noOfTrades,Time,Date
0,NIFTY,FUTIDXNIFTY24-02-2026XX0.00,FUTIDX,Index Futures,NIFTY 24-Feb-2026,24-Feb-2026,-,0,25700.3,115.6,...,0,8002605,2.054915e+11,2.054915e+11,2.054915e+11,25713,209503,123117,16:44,2026-02-23
1,NIFTY,FUTIDXNIFTY30-03-2026XX0.00,FUTIDX,Index Futures,NIFTY 30-Mar-2026,30-Mar-2026,-,0,25855.0,112.4,...,0,6173180,1.594704e+11,1.594704e+11,1.594704e+11,25713,151834,94972,16:44,2026-02-23
2,NIFTY,FUTIDXNIFTY28-04-2026XX0.00,FUTIDX,Index Futures,NIFTY 28-Apr-2026,28-Apr-2026,-,0,26005.1,106.0,...,0,293085,7.622232e+09,7.622232e+09,7.622232e+09,25713,10101,4509,16:44,2026-02-23


In [17]:
df.columns

Index(['underlying', 'identifier', 'instrumentType', 'instrument', 'contract',
       'expiryDate', 'optionType', 'strikePrice', 'lastPrice', 'change',
       'pChange', 'openPrice', 'highPrice', 'lowPrice', 'closePrice', 'volume',
       'totalTurnover', 'value', 'premiumTurnOver', 'underlyingValue',
       'openInterest', 'noOfTrades', 'Time', 'Date'],
      dtype='object')

In [22]:
reqcol = ['underlying', 'instrumentType', 'contract',
       'lastPrice', 'change',
       'pChange','highPrice', 'lowPrice',  'volume',
       'totalTurnover', 'value', 'premiumTurnOver', 'underlyingValue',
       'openInterest', 'noOfTrades', 'Time', 'Date']
df = df[reqcol]

In [23]:
df.head()

,underlying,instrumentType,contract,lastPrice,change,pChange,highPrice,lowPrice,volume,totalTurnover,value,premiumTurnOver,underlyingValue,openInterest,noOfTrades,Time,Date
0,NIFTY,FUTIDX,NIFTY 24-Feb-2026,25700.3,115.6,0.45,25779.9,25593.3,8002605,2.054915e+11,2.054915e+11,2.054915e+11,25713,209503,123117,16:00,2026-02-23
1,NIFTY,FUTIDX,NIFTY 30-Mar-2026,25855.0,112.4,0.44,25934.6,25758.5,6173180,1.594704e+11,1.594704e+11,1.594704e+11,25713,151834,94972,16:00,2026-02-23
2,NIFTY,FUTIDX,NIFTY 28-Apr-2026,26005.1,106.0,0.41,26078.0,25925.0,293085,7.622232e+09,7.622232e+09,7.622232e+09,25713,10101,4509,16:00,2026-02-23


## Using request instead of client.

In [2]:

starttime = datetime.now()
print("\2\n"+ " ******************  Script run started  at ...   ", starttime)
current_date = starttime.strftime('%Y-%m-%d')
main_url = 'https://www.nseindia.com'
#fno_url = f'https://www.nseindia.com/api/NextApi/apiClient/GetQuoteApi?functionName=getSymbolDerivativesData&symbol={encoded_stock}'
fno_url = "https://www.nseindia.com/api/liveEquity-derivatives?index=nse50_fut"
headers = {
    'User-Agent': 'Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) '
                  'Chrome/91.0.4472.124 Safari/537.36',
    'Referer': main_url,
    'Accept': 'text/html,application/xhtml+xml,application/xml;q=0.9,image/avif,image/webp,*/*;q=0.8',
    'Accept-Language': 'en-US,en;q=0.5'
}
session = requests.Session()
retry = Retry(
    total=2,
    backoff_factor=1,
    status_forcelist=[429, 500, 502, 503, 504],
    allowed_methods=["GET"]
)
adapter = HTTPAdapter(max_retries=retry)
session.mount("https://", adapter)
session.mount("http://", adapter)
try:
    session.get(main_url, headers=headers)
    time.sleep(1)
    response2 = session.get(fno_url, headers=headers)
    response2.raise_for_status()
    fno = response2.json()
    df = pd.json_normalize(fno['data'])
    current_time = starttime.strftime('%H:%M')
    # These lines below were likely causing the TabError
    df['Time'] = current_time
    df['Date'] = current_date
    df['Fetcgdate'], df['Fetchtime'] = fno['timestamp'].split()
except requests.exceptions.RequestException as e:
    print(f"An error occurred for : {e}")

#df.to_sql('spur', conn, if_exists='append', index=False)
#bank = df[nfcolumns]
endtime = datetime.now()
elapsed = endtime - starttime

# total seconds as float
total_seconds = elapsed.total_seconds()
# minutes and seconds
minutes = int(total_seconds // 60)
seconds = int(total_seconds % 60)
milliseconds = int((total_seconds - int(total_seconds)) * 1000)
reqcol = ['underlying', 'Date', 'Fetchtime','instrumentType', 'contract',
       'lastPrice', 'change',
       'pChange','highPrice', 'lowPrice',  'volume',
       'totalTurnover', 'value', 'premiumTurnOver', 'underlyingValue',
       'openInterest', 'noOfTrades', 'Time']
df = df[reqcol]

print(f"The time taken to complete the task is {minutes}:{seconds:02d}.{milliseconds:03d}")
df.head()


 ******************  Script run started  at ...    2026-02-27 12:17:49.901298
The time taken to complete the task is 0:01.212


,underlying,Date,Fetchtime,instrumentType,contract,lastPrice,change,pChange,highPrice,lowPrice,volume,totalTurnover,value,premiumTurnOver,underlyingValue,openInterest,noOfTrades,Time
0,NIFTY,2026-02-27,12:16:09,FUTIDX,NIFTY 30-Mar-2026,25421.0,-214.8,-0.84,25580.0,25407.0,2901730,7.393596e+10,7.393596e+10,7.393596e+10,25286.15,214174,44642,12:17
1,NIFTY,2026-02-27,12:16:09,FUTIDX,NIFTY 28-Apr-2026,25582.0,-211.0,-0.82,25742.7,25568.0,297115,7.613753e+09,7.613753e+09,7.613753e+09,25286.15,15758,4571,12:17
2,NIFTY,2026-02-27,12:16:09,FUTIDX,NIFTY 26-May-2026,25701.7,-210.8,-0.81,25850.1,25681.6,155545,4.003161e+09,4.003161e+09,4.003161e+09,25286.15,2663,2393,12:17


In [5]:
reqcol = ['underlying', 'instrumentType', 'contract',
       'lastPrice', 'change',
       'pChange','highPrice', 'lowPrice',  'volume',
       'totalTurnover', 'value', 'premiumTurnOver', 'underlyingValue',
       'openInterest', 'noOfTrades', 'Time', 'Fetchtime','Date']
df = df[reqcol]
df.head()

,underlying,instrumentType,contract,lastPrice,change,pChange,highPrice,lowPrice,volume,totalTurnover,value,premiumTurnOver,underlyingValue,openInterest,noOfTrades,Time,Fetchtime,Date
0,NIFTY,FUTIDX,NIFTY 30-Mar-2026,25658.0,-201.7,-0.78,25800.0,25629.1,2281825,5.863277e+10,5.863277e+10,5.863277e+10,25492.45,171092,35105,10:29,10:27:44,2026-02-24
1,NIFTY,FUTIDX,NIFTY 24-Feb-2026,25499.9,-205.3,-0.80,25640.4,25470.6,1561430,3.986885e+10,3.986885e+10,3.986885e+10,25492.45,143510,24022,10:29,10:27:44,2026-02-24
2,NIFTY,FUTIDX,NIFTY 28-Apr-2026,25831.7,-183.3,-0.70,25945.0,25801.0,165165,4.271301e+09,4.271301e+09,4.271301e+09,25492.45,11078,2541,10:29,10:27:44,2026-02-24


## Bank Nifty Futures Live data.

In [69]:
starttime = datetime.now()
print("\2\n"+ " ******************  Script run started  at ...   ", starttime)
current_date = starttime.strftime('%Y-%m-%d')
main_url = 'https://www.nseindia.com'
#fno_url = f'https://www.nseindia.com/api/NextApi/apiClient/GetQuoteApi?functionName=getSymbolDerivativesData&symbol={encoded_stock}'
#fno_url = "https://www.nseindia.com/api/liveEquity-derivatives?index=nse50_fut"
nbfu_url = "https://www.nseindia.com/api/liveEquity-derivatives?index=nifty_bank_fut"
headers = {
    'User-Agent': 'Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) '
                  'Chrome/91.0.4472.124 Safari/537.36',
    'Referer': main_url,
    'Accept': 'text/html,application/xhtml+xml,application/xml;q=0.9,image/avif,image/webp,*/*;q=0.8',
    'Accept-Language': 'en-US,en;q=0.5'
}
session = requests.Session()
retry = Retry(
    total=2,
    backoff_factor=1,
    status_forcelist=[429, 500, 502, 503, 504],
    allowed_methods=["GET"]
)
adapter = HTTPAdapter(max_retries=retry)
session.mount("https://", adapter)
session.mount("http://", adapter)
try:
    session.get(main_url, headers=headers)
    time.sleep(1)
    response2 = session.get(nbfu_url, headers=headers)
    response2.raise_for_status()
    fno = response2.json()
    df = pd.json_normalize(fno['data'])
    current_time = starttime.strftime('%H:%M')
    # These lines below were likely causing the TabError
    df['Time'] = current_time
    df['Date'] = current_date


except requests.exceptions.RequestException as e:
    print(f"An error occurred for : {e}")

#df.to_sql('spur', conn, if_exists='append', index=False)
#bank = df[nfcolumns]
reqcol = ['underlying', 'instrumentType', 'contract',
       'lastPrice', 'change',
       'pChange','highPrice', 'lowPrice',  'volume',
       'totalTurnover', 'value', 'premiumTurnOver', 'underlyingValue',
       'openInterest', 'noOfTrades', 'Time', 'Date']
df = df[reqcol]
endtime = datetime.now()
elapsed = endtime - starttime

# total seconds as float
total_seconds = elapsed.total_seconds()
# minutes and seconds
minutes = int(total_seconds // 60)
seconds = int(total_seconds % 60)
milliseconds = int((total_seconds - int(total_seconds)) * 1000)
print(f"The time taken to complete the task is {minutes}:{seconds:02d}.{milliseconds:03d}")


 ******************  Script run started  at ...    2026-02-23 16:39:02.336545
The time taken to complete the task is 0:01.351


In [70]:
df.head()

,underlying,instrumentType,contract,lastPrice,change,pChange,highPrice,lowPrice,volume,totalTurnover,value,premiumTurnOver,underlyingValue,openInterest,noOfTrades,Time,Date
0,BANKNIFTY,FUTIDX,BANKNIFTY 24-Feb-2026,61219.0,39.8,0.07,61500.0,61020.2,681450,4.171876e+10,4.171876e+10,4.171876e+10,61264.25,28727,22715,16:39,2026-02-23
1,BANKNIFTY,FUTIDX,BANKNIFTY 30-Mar-2026,61603.4,57.6,0.09,61850.0,61416.0,635610,3.915076e+10,3.915076e+10,3.915076e+10,61264.25,37365,21187,16:39,2026-02-23
2,BANKNIFTY,FUTIDX,BANKNIFTY 28-Apr-2026,61940.0,73.2,0.12,62175.8,61764.0,28080,1.739593e+09,1.739593e+09,1.739593e+09,61264.25,3077,936,16:39,2026-02-23


## Bank Nifty Options Live Data.

In [3]:
starttime = datetime.now()
print("\2\n"+ " ******************  Script run started  at ...   ", starttime)
current_date = starttime.strftime('%Y-%m-%d')
main_url = 'https://www.nseindia.com'
#n50fu_url = "https://www.nseindia.com/api/liveEquity-derivatives?index=nse50_fut"
#n50op_url = "https://www.nseindia.com/api/liveEquity-derivatives?index=nse50_opt"
#nbfu_url = "https://www.nseindia.com/api/liveEquity-derivatives?index=nifty_bank_fut"
nbop_url = "https://www.nseindia.com/api/liveEquity-derivatives?index=nifty_bank_opt"

headers = {
    'User-Agent': 'Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) '
                  'Chrome/91.0.4472.124 Safari/537.36',
    'Referer': main_url,
    'Accept': 'text/html,application/xhtml+xml,application/xml;q=0.9,image/avif,image/webp,*/*;q=0.8',
    'Accept-Language': 'en-US,en;q=0.5'
}
session = requests.Session()
retry = Retry(
    total=2,
    backoff_factor=1,
    status_forcelist=[429, 500, 502, 503, 504],
    allowed_methods=["GET"]
)
adapter = HTTPAdapter(max_retries=retry)
session.mount("https://", adapter)
session.mount("http://", adapter)
try:
    session.get(main_url, headers=headers)
    time.sleep(1)
    response2 = session.get(nbop_url, headers=headers)
    response2.raise_for_status()
    fno = response2.json()
    df = pd.json_normalize(fno['data'])
    current_time = starttime.strftime('%H:%M')
    # These lines below were likely causing the TabError
    df['Time'] = current_time
    df['Date'] = current_date


except requests.exceptions.RequestException as e:
    print(f"An error occurred for : {e}")

#df.to_sql('spur', conn, if_exists='append', index=False)
#bank = df[nfcolumns]
opcol = ['underlying', 'instrumentType', 'contract',
       'optionType', 'strikePrice', 'lastPrice', 'change',
       'pChange','highPrice', 'lowPrice', 'volume',
       'totalTurnover', 'value', 'premiumTurnOver', 'underlyingValue',
       'openInterest', 'noOfTrades', 'Time', 'Date']
df = df[opcol]
endtime = datetime.now()
elapsed = endtime - starttime

# total seconds as float
total_seconds = elapsed.total_seconds()
# minutes and seconds
minutes = int(total_seconds // 60)
seconds = int(total_seconds % 60)
milliseconds = int((total_seconds - int(total_seconds)) * 1000)
print(f"The time taken to complete the task is {minutes}:{seconds:02d}.{milliseconds:03d}")


 ******************  Script run started  at ...    2026-02-27 12:19:05.609054
The time taken to complete the task is 0:01.651


In [55]:
df.head()

,underlying,instrumentType,contract,optionType,strikePrice,lastPrice,change,pChange,highPrice,lowPrice,volume,totalTurnover,value,premiumTurnOver,underlyingValue,openInterest,noOfTrades,Time,Date
0,BANKNIFTY,OPTIDX,BANKNIFTY 24-Feb-2026,Put,61000,134.95,-102.85,-43.25,260.00,105.90,37859100,6.980839e+09,6.980839e+09,2.316386e+12,61264.25,42251.0,1261970,16:29,2026-02-23
1,BANKNIFTY,OPTIDX,BANKNIFTY 24-Feb-2026,Call,61300,175.35,-79.15,-31.10,379.90,128.95,27244950,6.086794e+09,6.086794e+09,1.676202e+12,61264.25,26395.0,908165,16:29,2026-02-23
2,BANKNIFTY,OPTIDX,BANKNIFTY 24-Feb-2026,Call,61200,225.00,-75.85,-25.21,447.45,164.40,26960400,7.018331e+09,7.018331e+09,1.656995e+12,61264.25,22479.0,898680,16:29,2026-02-23
3,BANKNIFTY,OPTIDX,BANKNIFTY 24-Feb-2026,Call,61500,103.00,-67.55,-39.61,268.15,78.15,26895870,3.882688e+09,3.882688e+09,1.657979e+12,61264.25,34797.0,896529,16:29,2026-02-23
4,BANKNIFTY,OPTIDX,BANKNIFTY 24-Feb-2026,Put,61200,213.20,-106.35,-33.28,367.10,152.00,26153250,6.898443e+09,6.898443e+09,1.607477e+12,61264.25,15870.0,871775,16:29,2026-02-23


In [52]:
df.columns

Index(['underlying', 'identifier', 'instrumentType', 'instrument', 'contract',
       'expiryDate', 'optionType', 'strikePrice', 'lastPrice', 'change',
       'pChange', 'openPrice', 'highPrice', 'lowPrice', 'closePrice', 'volume',
       'totalTurnover', 'value', 'premiumTurnOver', 'underlyingValue',
       'openInterest', 'noOfTrades', 'Time', 'Date'],
      dtype='object')

In [ ]:
opcol = ['underlying', 'instrumentType' 'contract',
       'optionType', 'strikePrice', 'lastPrice', 'change',
       'pChange','highPrice', 'lowPrice', 'volume',
       'totalTurnover', 'value', 'premiumTurnOver', 'underlyingValue',
       'openInterest', 'noOfTrades', 'Time', 'Date']

In [71]:
starttime = datetime.now()
print("\2\n"+ " ******************  Script run started  at ...   ", starttime)
current_date = starttime.strftime('%Y-%m-%d')
main_url = 'https://www.nseindia.com'
#n50fu_url = "https://www.nseindia.com/api/liveEquity-derivatives?index=nse50_fut"
#n50op_url = "https://www.nseindia.com/api/liveEquity-derivatives?index=nse50_opt"
#nbfu_url = "https://www.nseindia.com/api/liveEquity-derivatives?index=nifty_bank_fut"
nbop_url = "https://www.nseindia.com/api/liveEquity-derivatives?index=nifty_bank_opt"

headers = {
    'User-Agent': 'Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) '
                  'Chrome/91.0.4472.124 Safari/537.36',
    'Referer': main_url,
    'Accept': 'text/html,application/xhtml+xml,application/xml;q=0.9,image/avif,image/webp,*/*;q=0.8',
    'Accept-Language': 'en-US,en;q=0.5'
}
session = requests.Session()
retry = Retry(
    total=2,
    backoff_factor=1,
    status_forcelist=[429, 500, 502, 503, 504],
    allowed_methods=["GET"]
)
adapter = HTTPAdapter(max_retries=retry)
session.mount("https://", adapter)
session.mount("http://", adapter)
try:
    session.get(main_url, headers=headers)
    time.sleep(1)
    response2 = session.get(nbop_url, headers=headers)
    response2.raise_for_status()
    fno = response2.json()
    df = pd.json_normalize(fno['data'])
    df['Date'], df['Time'] = fno['timestamp'].split()

except requests.exceptions.RequestException as e:
    print(f"An error occurred for : {e}")

#df.to_sql('spur', conn, if_exists='append', index=False)
#bank = df[nfcolumns]
opcol = ['underlying', 'instrumentType', 'contract',
       'optionType', 'strikePrice', 'lastPrice', 'change',
       'pChange','highPrice', 'lowPrice', 'volume',
       'totalTurnover', 'value', 'premiumTurnOver', 'underlyingValue',
       'openInterest', 'noOfTrades', 'Time', 'Date']
df = df[opcol]
endtime = datetime.now()
elapsed = endtime - starttime

# total seconds as float
total_seconds = elapsed.total_seconds()
# minutes and seconds
minutes = int(total_seconds // 60)
seconds = int(total_seconds % 60)
milliseconds = int((total_seconds - int(total_seconds)) * 1000)
print(f"The time taken to complete the task is {minutes}:{seconds:02d}.{milliseconds:03d}")


 ******************  Script run started  at ...    2026-02-23 16:40:51.887317
The time taken to complete the task is 0:01.390


In [72]:
df.head()

,underlying,instrumentType,contract,optionType,strikePrice,lastPrice,change,pChange,highPrice,lowPrice,volume,totalTurnover,value,premiumTurnOver,underlyingValue,openInterest,noOfTrades,Time,Date
0,BANKNIFTY,OPTIDX,BANKNIFTY 24-Feb-2026,Put,61000,134.95,-102.85,-43.25,260.00,105.90,37859100,6.980839e+09,6.980839e+09,2.316386e+12,61264.25,42251.0,1261970,15:30:00,23-Feb-2026
1,BANKNIFTY,OPTIDX,BANKNIFTY 24-Feb-2026,Call,61300,175.35,-79.15,-31.10,379.90,128.95,27244950,6.086794e+09,6.086794e+09,1.676202e+12,61264.25,26395.0,908165,15:30:00,23-Feb-2026
2,BANKNIFTY,OPTIDX,BANKNIFTY 24-Feb-2026,Call,61200,225.00,-75.85,-25.21,447.45,164.40,26960400,7.018331e+09,7.018331e+09,1.656995e+12,61264.25,22479.0,898680,15:30:00,23-Feb-2026
3,BANKNIFTY,OPTIDX,BANKNIFTY 24-Feb-2026,Call,61500,103.00,-67.55,-39.61,268.15,78.15,26895870,3.882688e+09,3.882688e+09,1.657979e+12,61264.25,34797.0,896529,15:30:00,23-Feb-2026
4,BANKNIFTY,OPTIDX,BANKNIFTY 24-Feb-2026,Put,61200,213.20,-106.35,-33.28,367.10,152.00,26153250,6.898443e+09,6.898443e+09,1.607477e+12,61264.25,15870.0,871775,15:30:00,23-Feb-2026


In [73]:
df.shape

(894, 19)

## OI Spurts Underlying and Contracts
### https://www.nseindia.com/api/live-analysis-oi-spurts-underlyings
### https://www.nseindia.com/api/live-analysis-oi-spurts-contracts

In [8]:
main_url = 'https://www.nseindia.com'
#n50fu_url = "https://www.nseindia.com/api/liveEquity-derivatives?index=nse50_fut"
#n50op_url = "https://www.nseindia.com/api/liveEquity-derivatives?index=nse50_opt"
#nbfu_url = "https://www.nseindia.com/api/liveEquity-derivatives?index=nifty_bank_fut"
#oispurts_url = "https://www.nseindia.com/api/live-analysis-oi-spurts-underlyings"
#oisprcon_url = "https://www.nseindia.com/api/live-analysis-oi-spurts-contracts"
oispurts_url = "https://www.nseindia.com/api/live-analysis-oi-spurts-underlyings"

headers = {
    'User-Agent': 'Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) '
                  'Chrome/91.0.4472.124 Safari/537.36',
    'Referer': main_url,
    'Accept': 'text/html,application/xhtml+xml,application/xml;q=0.9,image/avif,image/webp,*/*;q=0.8',
    'Accept-Language': 'en-US,en;q=0.5'
}
session = requests.Session()
retry = Retry(
    total=2,
    backoff_factor=1,
    status_forcelist=[429, 500, 502, 503, 504],
    allowed_methods=["GET"]
)
adapter = HTTPAdapter(max_retries=retry)
session.mount("https://", adapter)
session.mount("http://", adapter)
try:
    session.get(main_url, headers=headers)
    time.sleep(1)
    response2 = session.get(oispurts_url, headers=headers)
    response2.raise_for_status()
    fno = response2.json()
    df = pd.json_normalize(fno['data'])
    df['Date'], df['Time'] = fno['timestamp'].split()

except requests.exceptions.RequestException as e:
    print(f"An error occurred for : {e}")
df.head()

,symbol,latestOI,prevOI,changeInOI,avgInOI,volume,futValue,optValue,total,premValue,underlyingValue,Date,Time
0,FINNIFTY,17291,11918,5373,45.08,27760,3152.51586,47013050793,7.720194e+03,4.567678e+03,27995,27-Feb-2026,12:25:18
1,NIFTY,10131055,7031502,3099553,44.08,69121410,894996.74806,114314901384671,4.225274e+06,3.330277e+06,25273,27-Feb-2026,12:25:18
2,LTF,44638,38310,6328,16.52,45240,94082.81400,21061692022,9.915965e+04,5.076833e+03,291,27-Feb-2026,12:25:18
3,POLYCAB,36439,32058,4381,13.67,33027,44611.18666,31362937608,5.139994e+04,6.788751e+03,8664,27-Feb-2026,12:25:18
4,360ONE,7180,6322,858,13.57,3645,5959.70625,1468981910,6.251125e+03,2.914191e+02,1103,27-Feb-2026,12:25:18


## Spurts Contracts.

In [14]:
main_url = 'https://www.nseindia.com'
#n50fu_url = "https://www.nseindia.com/api/liveEquity-derivatives?index=nse50_fut"
#n50op_url = "https://www.nseindia.com/api/liveEquity-derivatives?index=nse50_opt"
#nbfu_url = "https://www.nseindia.com/api/liveEquity-derivatives?index=nifty_bank_fut"
#oispurts_url = "https://www.nseindia.com/api/live-analysis-oi-spurts-underlyings"
#oisprcon_url = "https://www.nseindia.com/api/live-analysis-oi-spurts-contracts"
oisprcon_url = "https://www.nseindia.com/api/live-analysis-oi-spurts-contracts"

headers = {
    'User-Agent': 'Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) '
                  'Chrome/91.0.4472.124 Safari/537.36',
    'Referer': main_url,
    'Accept': 'text/html,application/xhtml+xml,application/xml;q=0.9,image/avif,image/webp,*/*;q=0.8',
    'Accept-Language': 'en-US,en;q=0.5'
}
session = requests.Session()
retry = Retry(
    total=2,
    backoff_factor=1,
    status_forcelist=[429, 500, 502, 503, 504],
    allowed_methods=["GET"]
)
adapter = HTTPAdapter(max_retries=retry)
session.mount("https://", adapter)
session.mount("http://", adapter)
try:
    session.get(main_url, headers=headers)
    time.sleep(1)
    response2 = session.get(oisprcon_url, headers=headers)
    response2.raise_for_status()
    fno = response2.json()
    df = pd.json_normalize(fno['data'])
    df['Date'], df['Time'] = fno['timestamp'].split()

except requests.exceptions.RequestException as e:
    print(f"An error occurred for : {e}")
df.head()

,Slide-in-OI-Slide,Slide-in-OI-Rise,Rise-in-OI-Rise,Rise-in-OI-Slide,Date,Time
0,"[{'type': 'SS', 'symbol': 'NIFTY', 'instrument...",NaN,NaN,NaN,23-Feb-2026,15:31:39
1,NaN,"[{'type': 'SR', 'symbol': 'NIFTY', 'instrument...",NaN,NaN,23-Feb-2026,15:31:39
2,NaN,NaN,"[{'type': 'RR', 'symbol': 'HDFCBANK', 'instrum...",NaN,23-Feb-2026,15:31:39
3,NaN,NaN,NaN,"[{'type': 'RS', 'symbol': 'NIFTY', 'instrument...",23-Feb-2026,15:31:39


In [5]:
dfdata = pd.json_normalize(fno['data'][0]['Slide-in-OI-Slide'])
dfdata.head()

KeyError: 'Slide-in-OI-Slide'

In [6]:
df2 = pd.json_normalize(fno['data'][1]['Slide-in-OI-Rise'])
df3 = pd.json_normalize(fno['data'][2]['Rise-in-OI-Rise'])
df4 = pd.json_normalize(fno['data'][3]['Rise-in-OI-Slide'])
df2.head()

KeyError: 'Slide-in-OI-Rise'

In [7]:
df3.head()

NameError: name 'df3' is not defined

In [41]:
df4.head()

,type,symbol,instrument,expiryDate,optionType,strikePrice,ltp,prevClose,pChange,latestOI,prevOI,changeInOI,volume,turnover,premTurnover,underlyingValue,identifier,instrumentType,pChangeInOI
0,RS,NIFTY,Index Options,24-Feb-2026,Put,25700,82.00,181.30,-54.77,171117,49925,121192,9877558,662714.99889,662714.99889,25713.0,OPTIDXNIFTY24-02-2026PE25700,OPTIDX,242.75
1,RS,NIFTY,Index Options,24-Feb-2026,Put,25600,47.10,128.80,-63.43,173084,98538,74546,9367333,402163.02402,402163.02402,25713.0,OPTIDXNIFTY24-02-2026PE25600,OPTIDX,75.65
2,RS,NIFTY,Index Options,24-Feb-2026,Call,26000,9.00,16.00,-43.75,285253,217495,67758,4775022,43483.73784,43483.73784,25713.0,OPTIDXNIFTY24-02-2026CE26000,OPTIDX,31.15
3,RS,NIFTY,Index Options,24-Feb-2026,Put,25650,62.15,149.75,-58.50,96826,36187,60639,8302078,458905.66352,458905.66352,25713.0,OPTIDXNIFTY24-02-2026PE25650,OPTIDX,167.57
4,RS,NIFTY,Index Options,24-Feb-2026,Call,25800,41.65,43.30,-3.81,225726,165418,60308,8720038,290939.70785,290939.70785,25713.0,OPTIDXNIFTY24-02-2026CE25800,OPTIDX,36.46


In [42]:
df4['instrumentType'].unique()

array(['OPTIDX', 'FUTSTK'], dtype=object)

## Indices Change code.

In [62]:
spurdb = os.path.join(BASE_DIR_db, "fuopspur.db")
conn = sqlite3.connect(spurdb)
query_spur = f"SELECT * FROM spur ORDER BY Date DESC , Time DESC"
spurdf = pd.read_sql_query(query_spur, conn)

# Assuming your DataFrame is named df
banknifty = [
    "HDFCBANK", "ICICIBANK", "SBIN", "KOTAKBANK", "AXISBANK",
    "FEDERALBNK", "CANBK", "BANKBARODA", "UNIONBANK", "PNB",
    "IDFCFIRSTB", "AUBANK", "INDUSINDBK", "YESBANK"
]

bn_df = spurdf[spurdf["symbol"].isin(banknifty)]


In [67]:
icici = bn[bn['symbol'] == 'ICICIBANK']
idfc = bn[bn['symbol'] == 'IDFCFIRSTB']
idfc.head()

,symbol,latestOI,prevOI,changeInOI,avgInOI,volume,futValue,optValue,total,premValue,underlyingValue,Time,Date
0,IDFCFIRSTB,111250,57575,53675,93.23,354934,537807.52116,191346674608,596061.68424,58254.16308,70,15:32,2026-02-23
212,IDFCFIRSTB,111322,57575,53747,93.35,352862,535712.09090,190126239037,593692.79452,57980.70362,70,15:30,2026-02-23
424,IDFCFIRSTB,111124,57575,53549,93.01,350458,532041.00323,188846978262,589773.07285,57732.06962,70,15:28,2026-02-23
636,IDFCFIRSTB,111193,57575,53618,93.13,349302,529725.53936,188271249146,587344.89733,57619.35796,70,15:26,2026-02-23
848,IDFCFIRSTB,111164,57575,53589,93.08,347763,527880.38849,187388585795,585268.06994,57387.68145,70,15:24,2026-02-23


In [68]:
idfc.columns

Index(['symbol', 'latestOI', 'prevOI', 'changeInOI', 'avgInOI', 'volume',
       'futValue', 'optValue', 'total', 'premValue', 'underlyingValue', 'Time',
       'Date'],
      dtype='object')

In [71]:
reqcol = ['symbol','Date','Time', 'latestOI','volume',
       'futValue', 'optValue', 'total', 'premValue', 'underlyingValue']
icici = icici[reqcol]    
icici.head()

,symbol,Date,Time,latestOI,volume,futValue,optValue,total,premValue,underlyingValue
45,ICICIBANK,2026-02-23,15:32,296774,232864,817069.96301,147681282889,822636.65190,5566.68889,1399
257,ICICIBANK,2026-02-23,15:30,296798,231279,813659.72975,146470407853,819196.30828,5536.57853,1398
468,ICICIBANK,2026-02-23,15:28,296561,230178,811311.03907,145626823006,816813.93913,5502.90006,1399
681,ICICIBANK,2026-02-23,15:26,296510,229167,806580.84514,145105614262,812059.70776,5478.86262,1399
893,ICICIBANK,2026-02-23,15:24,296400,228123,803021.93552,144438631736,808475.23288,5453.29736,1399


In [3]:
changecol = ['latestOI','volume',
       'futValue', 'optValue', 'total', 'premValue', 'underlyingValue']
stock = 'SBIN'

In [4]:
    db_path = os.path.join(BASE_DIR_db, "niftybank.db")
    todays_date = date.today()
    with sqlite3.connect(db_path) as tg_conn:
        tg_query = f"""
            SELECT * FROM niftybank
            WHERE Date = "{todays_date}" AND Stock = "{stock}"
            ORDER BY Time DESC
        """
        tgdf = pd.read_sql_query(tg_query, tg_conn)

    alldf = tgdf[tgdf['Stock'] == stock].sort_values(by=['Time'], ascending=False).reset_index(drop=True)

In [5]:
alldf.head()

,Stock,Date,Time,fetch_time,Tbuy_eq,Tsell_eq,Tvalue_eq,Tvol_eq,High_eq,PrChg_eq,...,Ltp_eq,Vwap_eq,Ltp_fu,PrChg_fu,%Prchg_fu,Tsell_fu,Tbuy_fu,Tvol_fu,Tvalue_fu,fuOI
0,SBIN,2026-02-26,10:11,10:09,408457,606451,1.543478e+09,1283803,1205.6,0.2,...,1200.1,1202.27,1207.5,0,0.00,867000,703500,3359,3045370170,68393
1,SBIN,2026-02-26,10:10,10:09,428457,589355,1.522615e+09,1266429,1205.6,1.5,...,1200.1,1202.29,1207.5,0,0.00,867000,703500,3359,3045370170,68393
2,SBIN,2026-02-26,10:09,10:07,421402,588078,1.500300e+09,1247858,1205.6,2.4,...,1200.1,1202.30,1209,1.5,0.12,832500,729000,3280,2973746400,68300
3,SBIN,2026-02-26,10:08,10:07,416745,578044,1.486928e+09,1236736,1205.6,1.9,...,1200.1,1202.30,1209,1.5,0.12,832500,729000,3280,2973746400,68300
4,SBIN,2026-02-26,10:07,10:05,410888,580371,1.479564e+09,1230611,1205.6,1.3,...,1200.1,1202.30,1208.2,0.7,0.06,834750,717750,3239,2936574570,68300


In [6]:
with sqlite3.connect(os.path.join(BASE_DIR_db, "niftybank.db")) as conn:
    fno_df = pd.read_sql_query("SELECT DISTINCT Stock FROM niftybank ORDER BY Stock", conn)


In [8]:
fno_df.head(21)

,Stock
0,AUBANK
1,AXISBANK
2,BANKBARODA
3,CANBK
4,FEDERALBNK
5,HDFCBANK
6,ICICIBANK
7,IDFCFIRSTB
8,INDUSINDBK
9,KOTAKBANK
